In [2]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np

## Define nodes

In [4]:
# Define nodes
station_codes = pd.read_csv('mrt_lrt_stations.csv')
# exclude all the stations that LINE_COLOR is not grey
nodes = station_codes[station_codes['LINE_COLOR'] != 'Grey']['ALPHANUMERIC_CODE'].unique().tolist()
nodes.insert(nodes.index('EW33') + 1, 'CG0') # insert CG0 because tanah merah has no CG code but it is an interchange to go to expo/CG
nodes.insert(nodes.index('CC29') + 1, 'CE0') # insert CE0 because promenade has no CE code but it is an interchange to go to sentosa/CE
print(nodes)

['NS1', 'NS2', 'NS3', 'NS4', 'NS5', 'NS7', 'NS8', 'NS9', 'NS10', 'NS11', 'NS12', 'NS13', 'NS14', 'NS15', 'NS16', 'NS17', 'NS18', 'NS19', 'NS20', 'NS21', 'NS22', 'NS23', 'NS24', 'NS25', 'NS26', 'NS27', 'NS28', 'EW1', 'EW2', 'EW3', 'EW4', 'EW5', 'EW6', 'EW7', 'EW8', 'EW9', 'EW10', 'EW11', 'EW12', 'EW13', 'EW14', 'EW15', 'EW16', 'EW17', 'EW18', 'EW19', 'EW20', 'EW21', 'EW22', 'EW23', 'EW24', 'EW25', 'EW26', 'EW27', 'EW28', 'EW29', 'EW30', 'EW31', 'EW32', 'EW33', 'CG0', 'CG1', 'CG2', 'NE1', 'NE3', 'NE4', 'NE5', 'NE6', 'NE7', 'NE8', 'NE9', 'NE10', 'NE11', 'NE12', 'NE13', 'NE14', 'NE15', 'NE16', 'NE17', 'CC1', 'CC2', 'CC3', 'CC4', 'CC5', 'CC6', 'CC7', 'CC8', 'CC9', 'CC10', 'CC11', 'CC12', 'CC13', 'CC14', 'CC15', 'CC16', 'CC17', 'CC19', 'CC20', 'CC21', 'CC22', 'CC23', 'CC24', 'CC25', 'CC26', 'CC27', 'CC28', 'CC29', 'CE0', 'CE1', 'CE2', 'DT1', 'DT2', 'DT3', 'DT5', 'DT6', 'DT7', 'DT8', 'DT9', 'DT10', 'DT11', 'DT12', 'DT13', 'DT14', 'DT15', 'DT16', 'DT17', 'DT18', 'DT19', 'DT20', 'DT21', 'DT22',

## Define arcs

In [ ]:
# each node is connected to the next node in the same line, and also to the interchanges
# the nodes have values e.g. NS1, NS2, NS3. NS1 has an arc to NS2, NS2 has an arc to NS3 but NS1 has no arc to NS3.
arcs = []
for i in range(len(nodes) - 1):
    if nodes[i][:2] == nodes[i + 1][:2]:
        arcs.append((nodes[i], nodes[i + 1]))
# print(arcs)
interchanges = [('NS1', 'EW24'), ('NS9', 'TE2'), ('CE0', 'CC4'), ('CE0', 'DT16'),
                ('NS17', 'CC15'), ('NS21', 'DT11'), ('NS22', 'TE14'), 
               ('NS24', 'NE6'), ('NS24', 'CC1'), ('NS25', 'EW13'), 
               ('NS26', 'EW14'), ('NS27', 'TE20'), ('NS27', 'CE2'),
               ('CC22', 'EW21'), ('EW16', 'NE3'), ('EW16', 'TE17'), 
               ('EW12', 'DT14'), ('EW8', 'CC9'), ('EW4', 'CG0'), 
               ('EW2', 'DT32'), ('CG1', 'DT35'), ('CC19', 'DT9'),
               ('DT10', 'TE11'), ('NE7', 'DT12'), ('DT15', 'CC4'),
               ('DT16', 'CE1'), ('DT19', 'NE4'), ('DT26', 'CC10'),
               ('CC29', 'NE1'), ('CC17', 'TE9'), ('CC13', 'NE12'),
               ('CC1', 'NE6'), ('CE2', 'TE20'), ('NE3', 'TE17'), 
               ]
for interchange in interchanges:
    arcs.append(interchange)
# since all arcs are bidirectional, need to include the other direction as well.
for arc in arcs.copy():
    arcs.append((arc[1], arc[0]))

print(arcs)

[('NS1', 'NS2'), ('NS2', 'NS3'), ('NS3', 'NS4'), ('NS4', 'NS5'), ('NS5', 'NS7'), ('NS7', 'NS8'), ('NS8', 'NS9'), ('NS9', 'NS10'), ('NS10', 'NS11'), ('NS11', 'NS12'), ('NS12', 'NS13'), ('NS13', 'NS14'), ('NS14', 'NS15'), ('NS15', 'NS16'), ('NS16', 'NS17'), ('NS17', 'NS18'), ('NS18', 'NS19'), ('NS19', 'NS20'), ('NS20', 'NS21'), ('NS21', 'NS22'), ('NS22', 'NS23'), ('NS23', 'NS24'), ('NS24', 'NS25'), ('NS25', 'NS26'), ('NS26', 'NS27'), ('NS27', 'NS28'), ('EW1', 'EW2'), ('EW2', 'EW3'), ('EW3', 'EW4'), ('EW4', 'EW5'), ('EW5', 'EW6'), ('EW6', 'EW7'), ('EW7', 'EW8'), ('EW8', 'EW9'), ('EW9', 'EW10'), ('EW10', 'EW11'), ('EW11', 'EW12'), ('EW12', 'EW13'), ('EW13', 'EW14'), ('EW14', 'EW15'), ('EW15', 'EW16'), ('EW16', 'EW17'), ('EW17', 'EW18'), ('EW18', 'EW19'), ('EW19', 'EW20'), ('EW20', 'EW21'), ('EW21', 'EW22'), ('EW22', 'EW23'), ('EW23', 'EW24'), ('EW24', 'EW25'), ('EW25', 'EW26'), ('EW26', 'EW27'), ('EW27', 'EW28'), ('EW28', 'EW29'), ('EW29', 'EW30'), ('EW30', 'EW31'), ('EW31', 'EW32'), ('EW3

## Define costs
* Taken from the dataset that has the timings between stations, including transfer timings
* Link: https://docs.google.com/spreadsheets/d/13CTjSiDdrjvIJvvCso10TWSP5Bf1kZrR77W638BVFkI/edit?gid=987939193#gid=987939193

In [9]:
station_timings_cost = pd.read_csv('station_costs.csv')
station_timings_cost.head()

,Arrival_Time,Station,travel_time,station_code
0,11:52:42,woodlands north,NaN,TE1
1,11:54:38,woodlands,0:01:56,TE2
2,11:57:10,woodlands south,0:02:32,TE3
3,12:01:35,springleaf,0:04:25,TE4
4,12:04:38,lentor,0:03:03,TE5


In [20]:
station_timings_cost = pd.read_csv('station_costs_no_interchanges.csv')
# Map each station to cost given the codes
costs = {}
# print(station_timings_cost.iloc[2]['station_code'])
for index, row in station_timings_cost.iterrows():
    station_code = row['station_code']
    # convert travel_time to a string
    row['travel_time'] = str(row['travel_time'])
    prev_station_code = station_timings_cost.iloc[index-1]['station_code'] if index >= 0 else None
    # convert travel_time from 0:04:25 to number of seconds, which is 4*60 + 25 = 265 seconds
    cost_in_seconds = row['travel_time'].split(':') if row['travel_time'] != '0:00:00' else [0, 0, 0]
    cost_in_seconds = int(cost_in_seconds[0]) * 3600 + (int(cost_in_seconds[1]) * 60 + int(cost_in_seconds[2]))
    if prev_station_code and station_code[:2] == prev_station_code[:2] and cost_in_seconds != 0: # if the station is on the same line as the previous station, then the cost is the travel time between them
        costs[(prev_station_code, station_code)] = cost_in_seconds
        costs[(station_code, prev_station_code)] = cost_in_seconds # since the arcs are bidirectional, the cost is the same in both directions
# print(costs)
# need to manually add the costs for transfer at interchanges
station_timings_cost_interchanges = pd.read_csv('station_costs_interchanges.csv')
station_timings_cost_interchanges = station_timings_cost_interchanges.dropna()
for index, row in station_timings_cost_interchanges.iterrows():
    station_code_1 = row['station_code_1']
    station_code_2 = row['station_code_2']
    # convert travel_time to a string
    row['travel_time'] = str(row['travel_time'])
    cost_in_seconds = row['travel_time'].split(':') if row['travel_time'] != '0:00:00' else [0, 0, 0]
    cost_in_seconds = int(cost_in_seconds[0]) * 3600 + (int(cost_in_seconds[1]) * 60 + int(cost_in_seconds[2]))
    costs[(station_code_1, station_code_2)] = cost_in_seconds
    costs[(station_code_2, station_code_1)] = cost_in_seconds # since the arcs are bidirectional, the cost is the same in both directions
print(costs)

{('TE1', 'TE2'): 116, ('TE2', 'TE1'): 116, ('TE2', 'TE3'): 152, ('TE3', 'TE2'): 152, ('TE3', 'TE4'): 265, ('TE4', 'TE3'): 265, ('TE4', 'TE5'): 183, ('TE5', 'TE4'): 183, ('TE5', 'TE6'): 139, ('TE6', 'TE5'): 139, ('TE6', 'TE7'): 129, ('TE7', 'TE6'): 129, ('TE7', 'TE8'): 164, ('TE8', 'TE7'): 164, ('TE8', 'TE9'): 200, ('TE9', 'TE8'): 200, ('TE9', 'TE11'): 217, ('TE11', 'TE9'): 217, ('TE11', 'TE12'): 190, ('TE12', 'TE11'): 190, ('TE12', 'TE13'): 122, ('TE13', 'TE12'): 122, ('TE13', 'TE14'): 128, ('TE14', 'TE13'): 128, ('TE14', 'TE15'): 142, ('TE15', 'TE14'): 142, ('TE15', 'TE16'): 90, ('TE16', 'TE15'): 90, ('TE16', 'TE17'): 112, ('TE17', 'TE16'): 112, ('TE17', 'TE18'): 102, ('TE18', 'TE17'): 102, ('TE18', 'TE19'): 119, ('TE19', 'TE18'): 119, ('TE19', 'TE20'): 109, ('TE20', 'TE19'): 109, ('TE20', 'TE22'): 164, ('TE22', 'TE20'): 164, ('TE22', 'TE23'): 198, ('TE23', 'TE22'): 198, ('TE23', 'TE24'): 125, ('TE24', 'TE23'): 125, ('TE24', 'TE25'): 132, ('TE25', 'TE24'): 132, ('TE25', 'TE26'): 113, 

## Capacities for each arc
* Should be the same for every connected line, because 1 train will traverse the entire line of the same color.
* if the arc is describing a tranfer arc, i.e. via manual walking e.g. jurong east green to red line, we can assume the capacities to be infinite (capacities here in this case refer to the number of people that can transfer from 1 line to another, which is technically much larger when transiting between lines than the fixed capacity of a single train transporting passengers for a single line)
* referencing this article: https://medium.com/@simpletan/deep-dive-of-singapores-rail-network-passenger-loads-e331d3b9626b

In [ ]:
capacities = {}

# Supply and demand at each node
* Distance-defined. To be simulated based on distance from CBD
* Considering only evening peak hour, so supply (i.e. people going into the MRT) is higher as we get closer to the various CBD locations.

In [21]:
supply = {
    1: GRB.INFINITY,
    2: 0,
    3: 0,
    4: 0
}
# just an example of supply. change later

# Create the model

In [ ]:
m = gp.model('minimize_cost_to_travel')

# Constraints

In [22]:
# flow = m.addVars(arcs,ub=[capacities[arc] for arc in arcs])
# selling_amount = m.addVar(name="selling_amount")

# # Flow conservation constraints at nodes 
# for i in supply:
#     if (i == 1):
#         m.addConstr(flow.sum(i, '*') == selling_amount)
#     elif (i == 5):
#         m.addConstr(flow.sum('*', i) == selling_amount)
#     else:
#         m.addConstr(flow.sum(i, '*') - flow.sum('*', i) == 0)


# Objective function: 
* Requires 3 inputs from the user, depending on how much the user values each item. we will ask for 3 weights, and all 3 weights must sum up to 100
* All weights must also be positive.

w1 * total travel time + w2 * transfer penalties + w3 * overcrowding penalties (utility penalty)

In [23]:
while (True):
    w1 = int(input("On a scale of 1-100, how much do you value total travel time? "))
    w2 = int(input("On a scale of 1-100, how much do you value number of transfers? "))
    w3 = int(input("on a scale of 1-100, how much do you value overcrowding? "))
    if (w1 + w2 + w3 == 100):
        break
    else:
        print("The weights must sum to 100. Please try again.")

# Set objective
# m.setObjective(w1*travelling_time + w2*number_of_transfers + w3*overcrowding, GRB.MINIMIZE)


# Optimizing the model
* print the solutions also, which will show the entire route to take from station A to station B.

In [ ]:
m.optimize()


# # Print the solution
# print("---------------------------------")
# print("Optimal flow and cost:")
# print(selling_amount.VarName, selling_amount)
# print(f"Total profit: {m.objVal}")